# Stage 3 — Verify the Output Embedding Distinction

**Core theoretical claim:** The circuit latent space contains information that
the output embedding space discards.

**Experimental design:**
For each image in the val set, create a *degraded* version by adding Gaussian noise.
A robust model will classify both correctly — they produce similar *output* embeddings.
But they take different *computational paths* through the model (different early-layer
feature processing) — so their *circuit* embeddings should be further apart.

**What we expect to see:**
- For the CTLS model: circuit distances are *larger* than output distances for
  clean/degraded pairs
- For the baseline: circuit distances may be closer to output distances (less
  structured trajectory space)
- Within-class intraclass distance ordering: Spearman correlation between output
  and circuit distances should be lower for CTLS than baseline (more distinct information)

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'  # <-- edit this once
    REPO_DIR = '/content/ctls'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
        os.system(f'pip install -r {REPO_DIR}/requirements.txt -q')
    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f'Working directory: {os.getcwd()} ({"Colab" if IN_COLAB else "local"})')


In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders

def load_model(config_path, checkpoint_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    mcfg = config['model']
    ecfg = mcfg['meta_encoder']
    tcfg = config['training']
    soft_mask = SoftMask(init_temperature=tcfg['temperature']['init']).to(device)
    backbone = CTLSBackbone(
        arch=mcfg['arch'], num_classes=mcfg['num_classes'],
        soft_mask=soft_mask, pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()
    return backbone, meta_encoder, config

backbone_ctls, meta_enc_ctls, config = load_model(
    'configs/ctls.yaml', 'experiments/ctls/best.pt', DEVICE
)
backbone_base, meta_enc_base, base_config = load_model(
    'configs/baseline.yaml', 'experiments/baseline/best.pt', DEVICE
)

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=dcfg['batch_size'],
    num_workers=dcfg.get('num_workers', 4), augment=False, download=True,
)
print('Models loaded.')

## 1. Clean vs Degraded Distance Comparison

For each val image, add Gaussian noise (std=0.3, roughly 10–15% mean pixel noise)
and measure cosine distances in both spaces.

In [ ]:
from evaluation.embedding_compare import EmbeddingComparator

NOISE_STD = 0.3   # adjust to taste — higher = more aggressive degradation
N_SAMPLES  = 1024

comp_ctls = EmbeddingComparator(backbone_ctls, meta_enc_ctls, DEVICE)
comp_base = EmbeddingComparator(backbone_base, meta_enc_base, DEVICE)

print('Running CTLS comparison...')
results_ctls = comp_ctls.compare_clean_vs_degraded(
    val_loader, noise_std=NOISE_STD, n_samples=N_SAMPLES
)
print('Running Baseline comparison...')
results_base = comp_base.compare_clean_vs_degraded(
    val_loader, noise_std=NOISE_STD, n_samples=N_SAMPLES
)

print()
print(f'=== Clean vs Degraded Distances (noise_std={NOISE_STD}) ===')
print(f"{'Metric':<45} {'Baseline':>10} {'CTLS':>10}")
print('-' * 65)
for key in ['output_dist_mean', 'circuit_dist_mean', 'ratio_circuit_over_output']:
    print(f"{key:<45} {results_base[key]:>10.4f} {results_ctls[key]:>10.4f}")

**Interpretation:**
- `ratio_circuit_over_output > 1` means circuit space sees more difference between
  clean and noisy versions than output space does
- A higher ratio for CTLS than Baseline confirms the circuit space carries extra information

## 2. Distance Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Stage 3: Circuit vs Output Space — clean/degraded pair distances', fontsize=14)

def _plot_row(axes_row, results, title):
    out_d   = results['output_dists']
    circ_d  = results['circuit_dists']

    axes_row[0].hist(out_d, bins=40, alpha=0.7, color='steelblue', label='output space')
    axes_row[0].hist(circ_d, bins=40, alpha=0.7, color='darkorange', label='circuit space')
    axes_row[0].set_xlabel('Cosine distance')
    axes_row[0].set_ylabel('Count')
    axes_row[0].set_title(f'{title} — Histograms')
    axes_row[0].axvline(out_d.mean(), color='steelblue', linestyle='--', lw=1.5,
                        label=f'output mean={out_d.mean():.3f}')
    axes_row[0].axvline(circ_d.mean(), color='darkorange', linestyle='--', lw=1.5,
                        label=f'circuit mean={circ_d.mean():.3f}')
    axes_row[0].legend(fontsize=8)

    axes_row[1].scatter(out_d, circ_d, s=4, alpha=0.3, color='purple')
    lim = max(out_d.max(), circ_d.max()) * 1.05
    axes_row[1].plot([0, lim], [0, lim], 'k--', lw=0.8, label='y=x')
    axes_row[1].set_xlabel('Output space distance')
    axes_row[1].set_ylabel('Circuit space distance')
    axes_row[1].set_title(f'{title} — Per-pair scatter')
    axes_row[1].legend(fontsize=8)

_plot_row(axes[0], results_ctls, 'CTLS')
_plot_row(axes[1], results_base, 'Baseline')

fig.tight_layout()
os.makedirs('experiments/ctls', exist_ok=True)
fig.savefig('experiments/ctls/stage3_distance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Noise Level Sensitivity

Sweep across noise levels to see how the ratio changes. We expect the circuit-over-output
ratio to increase with noise for CTLS (the model routes more differently under degradation),
but stay flatter for the baseline.

In [ ]:
noise_levels = [0.05, 0.1, 0.2, 0.3, 0.5, 0.8]
ratios_ctls, ratios_base = [], []

for std in noise_levels:
    r_ctls = comp_ctls.compare_clean_vs_degraded(val_loader, noise_std=std, n_samples=256)
    r_base = comp_base.compare_clean_vs_degraded(val_loader, noise_std=std, n_samples=256)
    ratios_ctls.append(r_ctls['ratio_circuit_over_output'])
    ratios_base.append(r_base['ratio_circuit_over_output'])
    print(f'  noise_std={std:.2f} | CTLS ratio={r_ctls["ratio_circuit_over_output"]:.3f} | '
          f'Base ratio={r_base["ratio_circuit_over_output"]:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(noise_levels, ratios_ctls, 's-', color='darkorange', lw=2, label='CTLS')
ax.plot(noise_levels, ratios_base, 'o--', color='steelblue', lw=2, label='Baseline')
ax.axhline(1.0, color='gray', linestyle='--', lw=0.8, label='ratio=1 (no distinction)')
ax.set_xlabel('Noise std')
ax.set_ylabel('Circuit / output distance ratio')
ax.set_title('Circuit space captures degradation information output space misses')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/ctls/stage3_noise_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Intraclass Distance Rank Correlation

For each class, compute all pairwise cosine distances between val images in both
the output space and circuit space. Report Spearman ρ.

**Low ρ** = circuit space provides a fundamentally different ordering than output
space — they capture different aspects of intraclass variation.

**High ρ** = circuit and output spaces are redundant.

In [ ]:
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

print('CTLS intraclass rank correlations:')
rank_ctls = comp_ctls.intraclass_distance_rank(val_loader, n_samples=2000)
for cls, stats in rank_ctls.items():
    print(f'  {CIFAR10_CLASSES[cls]:<12} ρ={stats["spearman_rho"]:+.3f}  p={stats["p_value"]:.4f}')

print()
print('Baseline intraclass rank correlations:')
rank_base = comp_base.intraclass_distance_rank(val_loader, n_samples=2000)
for cls, stats in rank_base.items():
    print(f'  {CIFAR10_CLASSES[cls]:<12} ρ={stats["spearman_rho"]:+.3f}  p={stats["p_value"]:.4f}')

In [ ]:
classes = sorted(rank_ctls.keys())
rho_ctls = [rank_ctls[c]['spearman_rho'] for c in classes]
rho_base = [rank_base[c]['spearman_rho'] for c in classes]
names    = [CIFAR10_CLASSES[c] for c in classes]

x = np.arange(len(classes))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w/2, rho_base, w, label='Baseline', color='steelblue', alpha=0.8)
ax.bar(x + w/2, rho_ctls, w, label='CTLS', color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('Spearman ρ (circuit vs output ordering)')
ax.set_title('Intraclass rank correlation: lower = circuit space more distinct from output')
ax.axhline(0, color='black', lw=0.8)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
fig.savefig('experiments/ctls/stage3_rank_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Evidence that circuit space ≠ output space:**

| Test | Expected result | Observed |
|------|----------------|----------|
| Clean/degraded ratio > 1 (CTLS) | Circuit sees more than output | ___ |
| Clean/degraded ratio > baseline's | CTLS circuit more sensitive | ___ |
| Noise sweep: ratio increases with σ | Degradation captured more | ___ |
| Intraclass ρ: CTLS < Baseline | Circuit ordering more distinct | ___ |

**Next:** Run `stage4_ablation.ipynb` to validate depth-weighting design choice.